# Day 5.5 — Events, Logs and Checkpoints
Two records that are easy to confuse. **Events** are append-only observations: they explain
what happened and are never edited. A **checkpoint** is mutable continuation state: it is
what a paused run needs in order to carry on later, and it is deleted once it is used.

This lesson also makes the runtime's retry budget visible, because "the model call failed"
is the most common thing that will ever happen to your agent.


## Before you begin

### Learning outcomes

- Write a durable JSONL event log and read it back.
- Resume a paused run after rebuilding the runtime from scratch.
- Watch a bounded retry with exponential backoff, and see it give up.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

The event file survives on disk; a rebuilt runtime resumes the paused approval; a provider that fails once is retried and succeeds, and one that always fails stops after a fixed number of attempts.


## Concept briefing

## Events and checkpoints

Events are append-only observations such as run started, model completed, policy decided
and tool completed. A trace groups events belonging to one run. A checkpoint stores
continuation state so a paused run can resume.

A checkpoint is not an audit log, and an event log is not enough to resume execution.
Durable approval needs the exact pending action and a stable run identifier. Sensitive
arguments should be redacted or omitted from telemetry where possible.

## Retries, timeouts and retry budgets

Network calls fail. The runtime applies a timeout and retries transient failures
such as temporary rate limits, with **exponential backoff**: each attempt waits
twice as long as the previous one. Every attempt is recorded as a `provider_retry`
event, and the budget is bounded by `MAX_PROVIDER_RETRIES`; exhausting it emits
`provider_retry_budget_exhausted` and ends the run. In production, a small random
`jitter` is added to the delay so many clients do not all retry at the same instant.

Do not retry every failure. Invalid arguments, unknown tools and most configuration
or authentication errors will not improve on repetition, so the runtime records
`provider_error_not_retried` and stops at once. Consequential tools need an
idempotency strategy before any automatic retry. Step budget and retry budget are
separate limits so one failing provider cannot consume unlimited time or credit.

## Cost attribution

Record model, configuration version, input tokens, output tokens, reasoning tokens,
estimated cost and run ID. That makes it possible to compare agents and enforce
classroom budgets. Cost belongs to the complete run, including retries, not only
the final response - which is why usage is attached to every `model_completed`
event rather than to the result.

Mock mode reports empty usage on purpose: nothing was bought, so nothing is
counted. The included API credit is a controlled learning resource. Use mock mode
while debugging application logic; spend credit only when model behaviour itself
is the subject of the exercise.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — A fresh, disposable run folder

Every artefact this notebook writes goes under `data/generated/`, which the repository
ignores. A timestamped folder means re-running the notebook never collides with itself.


In [ ]:
from datetime import datetime

RUN_ROOT = PROJECT_ROOT / "data" / "generated" / ("events_lesson_" +
            datetime.now().strftime("%Y%m%d_%H%M%S"))
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("This run writes only inside:", RUN_ROOT)
print("Nothing outside data/generated/ is touched, so the lesson is safely repeatable.")

## Step 2 — Events go to a file as well as to memory

Give `EventStore` a path and every event is appended to a JSON-lines file the moment it
happens. Crash the process and the record of what happened is still on disk.


In [ ]:
from mini_harness import (EventStore, HarnessRuntime, JSONCheckpointStore,
                          MockModel, build_demo_registry, effective_step_limit)

events = EventStore(RUN_ROOT / "events.jsonl")
checkpoints = JSONCheckpointStore(RUN_ROOT / "checkpoints")
runtime = HarnessRuntime(build_demo_registry(), MockModel(), events, checkpoints)

task = load_config("task_agent")
print("Effective step limit for this run:", effective_step_limit(task))

paused = runtime.run(task, "Send the synthetic update")
print("Status:", paused.status)
print()
for event in events.get(paused.run_id):
    print(f"{event['event']:<20}", event["details"])

In [ ]:
# The same events are already durable on disk, one JSON object per line.
import json

lines = (RUN_ROOT / "events.jsonl").read_text(encoding="utf-8").splitlines()
print("Lines written to events.jsonl:", len(lines))
print()
print("First line, raw:")
print(" ", lines[0])
print()
print("Parsed back into Python:")
first = json.loads(lines[0])
print("  run_id   :", first["run_id"])
print("  event    :", first["event"])
print("  timestamp:", first["timestamp"])

## Step 3 — The checkpoint is what makes resuming possible

An event log tells you a run paused. It is not enough to *continue* it. The checkpoint
holds the exact pending action.


In [ ]:
state = checkpoints.load(paused.run_id)
print("Checkpoint contents:")
for key in sorted(state):
    value = state[key]
    shown = f"<{len(value)} history messages>" if key == "history" else value
    print(f"  {key:<12}:", shown)

print()
print("On disk at:", RUN_ROOT / "checkpoints" / f"{paused.run_id}.json")

In [ ]:
# Simulate a restart: throw the runtime away and build a brand-new one that shares
# only the folder on disk. The pending approval must survive that.
restarted = HarnessRuntime(build_demo_registry(), MockModel(),
                           events, JSONCheckpointStore(RUN_ROOT / "checkpoints"))

print("New runtime object:", id(restarted) != id(runtime))
done = restarted.resume(paused.run_id, task, approved=True)
print("Resumed status    :", done.status)
print("Tool output       :", done.output)
print("Checkpoint after  :", checkpoints.load(paused.run_id), "(consumed, so it cannot replay)")
print()
print("Full event trace across BOTH runtime objects:")
print(" ", [e["event"] for e in events.get(paused.run_id)])

## Step 4 — A transient failure is retried, with backoff

`FlakyModel` is a teaching double: it raises on its first call and then behaves like the
normal mock. Watch the runtime absorb that without the run failing.


In [ ]:
from mini_harness import FlakyModel, MAX_PROVIDER_RETRIES

print("Retry budget per model call:", MAX_PROVIDER_RETRIES, "extra attempts")

retry_events = EventStore()
flaky = HarnessRuntime(build_demo_registry(), FlakyModel(failures=1), retry_events)
recovered = flaky.run(load_config("research_agent"), "What is a harness?")

print("Final status:", recovered.status)
print()
for event in recovered.events:
    if event["event"].startswith("provider") or event["event"] in {"model_completed", "run_completed"}:
        print(f"{event['event']:<28}", event["details"])
print()
print("Note retry_in_seconds doubles each attempt. That is exponential backoff:")
print("  attempt 1 waits 0.05s, attempt 2 waits 0.10s, attempt 3 waits 0.20s ...")
print("Production code also adds a small random 'jitter' so that many clients")
print("recovering from the same outage do not all retry at the same instant.")

## Step 5 — The budget is bounded, and some errors are not retried at all

Retrying forever is just a slower outage. And retrying the *wrong* errors wastes time and
credit on something that cannot improve.


In [ ]:
always_failing = HarnessRuntime(build_demo_registry(), FlakyModel(failures=99), EventStore())
gave_up = always_failing.run(load_config("research_agent"), "What is a harness?")

print("Status:", gave_up.status)
print("Attempts recorded:", len([e for e in gave_up.events if e["event"] == "provider_retry"]) + 1)
for event in gave_up.events:
    if event["event"].startswith(("provider", "run_failed")):
        print(f"  {event['event']:<34}", event["details"].get("error", ""))

print()
# A ValueError means bad arguments or bad configuration. Repeating it changes nothing.
class MisconfiguredModel:
    def decide(self, prompt, config, tools, history):
        raise ValueError("model name is not valid for this provider")

not_retried = HarnessRuntime(build_demo_registry(), MisconfiguredModel(), EventStore())
result = not_retried.run(load_config("research_agent"), "hello")
print("Non-transient error status:", result.status)
print("Events:", [e["event"] for e in result.events])
print("-> provider_error_not_retried: the harness stops immediately instead of burning")
print("   the retry budget on an error that will never succeed.")

## Step 6 — Where cost lives

Cost belongs to a whole run — including the retries you just watched — not to the final
reply. That is why usage is attached to every `model_completed` event.


In [ ]:
totals = {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}
for event in recovered.events:
    if event["event"] == "model_completed":
        for key in totals:
            totals[key] += event["details"]["usage"].get(key, 0)

print("Model calls in that run:",
      len([e for e in recovered.events if e["event"] == "model_completed"]))
print("Totals attributed to run", recovered.run_id[:8], ":", totals)
print()
print("Zero, because this was MOCK mode - nothing was bought. In LIVE mode the same")
print("three lines give you a per-run bill, and because failed attempts also emit")
print("events you can see what an outage actually cost you.")

### Try it yourself

Predict: if you delete the checkpoint file before resuming, what does `resume()` do?


In [ ]:
# --- Worked solution ---
# Pause a fresh run, delete its checkpoint, then try to resume it.
victim = runtime.run(task, "Send another synthetic update")
path = RUN_ROOT / "checkpoints" / f"{victim.run_id}.json"
print("Paused          :", victim.status)
print("Checkpoint file :", path.name, "exists:", path.exists())

path.unlink()                       # simulate losing the durable state
print("Deleted it. exists:", path.exists())

outcome = runtime.resume(victim.run_id, task, approved=True)
print()
print("Status:", outcome.status)
print("Reason:", outcome.output)
print()
print("It fails closed. Without the exact pending action there is nothing safe to run,")
print("and the harness will not ask the model to invent it again.")
print("The EVENT log still shows the run paused - events explain, checkpoints continue.")

### Checkpoint

**1. You have the full event log for a paused run. Can you resume it?**

<details><summary>Show answer</summary>

No. The event log explains what happened; it is an audit record, and it deliberately does not promise to hold everything needed to continue. Resuming needs the checkpoint: the exact tool, the exact arguments and the conversation history, stored under a stable run id.

</details>

**2. Why does the runtime retry a `RuntimeError` but not a `ValueError`?**

<details><summary>Show answer</summary>

A `RuntimeError` from the provider means the network call itself failed - a timeout or a rate limit - and the same request may well succeed a moment later. A `ValueError` means the request was malformed or misconfigured; sending it again produces the identical error, so the harness records `provider_error_not_retried` and stops.

</details>

### Recap

- Limitation: printed output disappears with the kernel, and a paused run had nothing durable to continue from.
- Layer added: a JSONL event log, a JSON checkpoint store, and a bounded retry budget with exponential backoff, all recorded as events.
- Evidence: a brand-new runtime resumed a paused approval from disk; a single transient failure was retried and recovered; a permanently failing provider stopped after a fixed number of attempts.
